# If CO2 jumped to four times, how warm would Earth get, and when?

**EPS 88 · PyEarth.** This week runs on **Google Colab**, not DataHub: DataHub
cannot run PyTorch, which trains the networks.

1. **Open it.** [Click here to open this notebook in Colab](https://colab.research.google.com/github/AI4EPS/EPS88_PyEarth/blob/main/docs/notebooks/12_climate_pinn.ipynb) and sign in with a Google
   account (your Berkeley one works).
2. **Save your own copy first.** Choose **File ▸ Save a copy in Drive**. A new tab opens, called
   "Copy of …": work only in that tab. Colab does not keep changes made in the first one. Your copy
   lives in Google Drive, in the folder **Colab Notebooks**; open it from there next time.
3. **Keep the runtime Colab gives you (CPU).** Do not switch to a GPU: for networks this small it is
   slower, and it changes the numbers.
4. **Expect waits.** A cell that trains a network takes from a few seconds to about a minute, and
   the text above each says how long; the whole notebook takes about 1 minute of computing. While a
   cell runs, a spinning circle shows beside it: wait for it to stop.
5. **If Colab disconnects** (after a long pause, or when you close the tab), your code is still in
   your copy, but the notebook forgets what it ran. Choose **Runtime ▸ Run all** to rebuild it.

Suppose the carbon dioxide in the air jumped overnight to four times what it was
before industry, and stayed there. How warm would the Earth get, and how fast? Nobody can run that
experiment on the real planet, so climate scientists run it in their **climate models**.

Today you get the first 20 years of one such run and predict the next 130. A neural network fitted
to the 20 years fails after them. A network that must also obey the law of energy balance finds
numbers for that law, which is hard to do by hand, and follows the 130 years it never saw.
Then we check *why* it got them right, and find that its law is too simple. The point is the idea
of a **physics-informed network**, on a problem where every answer can be checked.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by a cell that already has the shape of the answer written in, with `...` wherever your
code goes. Replace every `...`, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
To submit from Colab: choose **Runtime ▸ Restart session and run all**, wait until the
last cell has run, then **File ▸ Download ▸ Download .ipynb**, and upload that `.ipynb` file to
Gradescope.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Runtime ▸ Restart session and run all** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say what the energy imbalance is and why it shrinks as the Earth warms; explain
why the surface warms fast, then slowly, with a deep ocean that takes heat for a long time; say why
a right warming from a too-simple equation can hide a wrong mechanism.

**The code.** Gradient descent in PyTorch · a network with `nn.Linear`, `nn.Tanh` and
`nn.Sequential` · a curve's slope with `torch.autograd.grad` · a loss that adds an equation's miss
to the data's.

**Eleven places where you write something: eight in class, three at home**, each headed *Your turn*.

The questions of the class:

1. What happens when CO2 jumps to four times?
2. Can a network predict the next 130 years from the first 20?
3. Can a network that obeys the energy balance do better?
4. Can we trust what the network learned?
5. Where do physics-informed networks earn their place?

## Setup

Run the next cell once. It reads two tables, one row per year for the 150 years after CO2 was
quadrupled in a climate model: `cesm2`, from [**CESM2**](https://www.cesm.ucar.edu/models/cesm2)
(the US National Center for Atmospheric Research), for class, and `mpi`, from
[**MPI-ESM1.2-LR**](https://www.mpimet.mpg.de/en/science/models/mpi-esm) (the Max Planck Institute
for Meteorology, Germany), for the homework. Each has three columns:

- `year`: years since CO2 was quadrupled, 1 to 150; each row is the average of that year;
- `warming_c`: the global surface air temperature, minus the same model's without the extra CO2, °C;
- `imbalance_w_m2`: the energy imbalance at the top of the atmosphere, also as a difference, in
  watts per square metre (W m⁻²); section 1 explains it.

These are **numbers made by a computer model, not measurements of the real Earth**, from
[CMIP6](https://pcmdi.llnl.gov/CMIP6/), the project in which modelling centres run the same
experiments so their models can be compared.

In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.linear_model import LinearRegression
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

# 150 model years after CO2 is quadrupled at once. Global means made from CMIP6 abrupt-4xCO2 and
# piControl output (CC BY 4.0): CESM2 (Danabasoglu et al., NCAR) and MPI-ESM1.2-LR (Wieners et
# al., MPI-M), via the Pangeo copy of CMIP6 on Google Cloud.
cesm2 = pd.read_csv(CACHE + "/week12c_cesm2_4x.csv")
mpi = pd.read_csv(CACHE + "/week12c_mpi_4x.csv")
print(len(cesm2), "years of CESM2;", len(mpi), "years of MPI-ESM1.2-LR")

## 1. What happens when CO2 jumps to four times?

**We cannot quadruple the real Earth's CO2 to see what happens, so this experiment is run inside
climate models.** **Climate model.** A computer program that steps the physics of the air, the ocean, ice and land forward in time, on a grid around the globe; its output is a simulation, not a measurement. In the run called *abrupt-4xCO2*, a model starts from the climate before
industry, and at year 0 its CO2 is multiplied by four and held there for 150 years. Nobody expects
CO2 to jump like this; it is a test, one simple push whose response is easy to see.

**Watch:** [*A Short Introduction to Climate Models: CMIP and CMIP6*](https://www.wcrp-climate.org/wgcm-cmip/cmip-video),
a short video from the World Climate Research Programme, on what a climate model is and how
modelling centres run the same experiments to compare their models.

**We hide most of the run.** The methods may learn only from the first 20 years; the other 130 are
kept to check each prediction. The next cell keeps the first 20 years as `first_years` and draws them.

In [ ]:
n_train = 20                                     # the years the methods may learn from
first_years = cesm2[cesm2["year"] <= n_train]    # those years
later = cesm2[cesm2["year"] > n_train]           # the hidden years after them, kept for the tests

plt.figure(figsize=(7, 4))
plt.scatter(first_years["year"], first_years["warming_c"], color="black", s=14)
plt.xlabel("years after CO2 was quadrupled")
plt.ylabel("global warming (°C)")
plt.title(f"CESM2: the first {n_train} years after CO2 was quadrupled")
plt.show()
print("warming in year 1:", round(first_years["warming_c"].iloc[0], 2), "°C;  year 10:", round(first_years["warming_c"].iloc[9], 2),
      f"°C;  year {n_train}:", round(first_years["warming_c"].iloc[n_train - 1], 2), "°C")

**Fast at first, then slower.** CESM2's Earth warms 0.87 °C in the first year and 3.96 °C by
year 10, but only 4.87 °C by year 20. Is it about to stop, or will it keep going? That is the
week's question: how warm, and when.

**Energy in, energy out.** The Earth is warmed by sunlight. It reflects some of it, and sends the
rest back to space as infrared heat. When the two balance, the temperature stays where it is.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week12c/energy_budget_ghe.png" width="460">

*The Earth's energy flows today, in watts per square metre averaged over the globe. At the top:
240 of sunlight absorbed (yellow) and 239 of infrared heat leaving for space (red). The 1 left over,
"net absorbed" at the bottom left, is today's energy imbalance: the Earth is gaining heat. The
greenhouse effect is the 159 that the surface emits but the atmosphere holds back. Diagram: R.
Wentworth, from IPCC AR6 numbers (CC BY-SA 4.0), via Wikimedia Commons.*

**Energy imbalance.** N: the sunlight the Earth absorbs minus the heat it sends back to space, in watts per square metre; while N is above zero the Earth keeps gaining heat and keeps warming. Think of a bathtub with the tap on and the plug
out: the water rises until it drains as fast as it pours in. CO2 holds back some outgoing heat, so
when it jumps, the imbalance jumps. The next cell draws it.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(first_years["year"], first_years["imbalance_w_m2"], color="tab:red", s=14)
plt.axhline(0, color="0.5", lw=0.8)
plt.xlabel("years after CO2 was quadrupled")
plt.ylabel("energy imbalance N (W per m²)")
plt.title(f"CESM2: the energy imbalance, first {len(first_years)} years")
plt.show()
print("imbalance in year 1:", round(first_years["imbalance_w_m2"].iloc[0], 2), "W per m²;  year", n_train, ":",
      round(first_years["imbalance_w_m2"].iloc[n_train - 1], 2), "W per m²")

**The imbalance shrinks as the Earth warms.** It starts at 7.69 W m⁻² and is down to
2.9 W m⁻² by year 20: a warmer Earth sends more heat to space, as a fuller bathtub drains
faster. The warming stops when the imbalance reaches zero.

In this notebook $T$ always means the **surface warming** in °C, and $N$ the imbalance. The
simplest picture (Gregory and others, 2004) says $N$ falls in a straight line as $T$ grows:

$$N = F - \lambda\,T$$

- **Radiative forcing.** F: the energy imbalance a change such as more CO2 causes before the Earth has warmed at all, in watts per square metre. It is where the line starts, at $T = 0$;
- **Climate feedback parameter.** λ: how many extra watts per square metre a warmer Earth sends back to space for each degree it has warmed. (λ, lambda.) It is minus
  the slope of the line.

If the line holds all the way, the warming stops where $N = 0$, that is at $T = F / \lambda$.

### ✏️ Your turn 1

**Where does the imbalance reach zero?**

**Why:** this gives *how warm*, with the straight-line fit you already know, before any network.

1. **`line`**: a straight line fitted on `first_years`, as in the regression week:
   `LinearRegression().fit(table[["input column"]], table["answer column"])`. The input is the
   warming, `"warming_c"` (in double brackets: the input is a table of columns); the answer is the
   imbalance, `"imbalance_w_m2"` (single brackets: one column).
2. `F_line`: $F$, the line's intercept, and `slope`, its slope (both given); `lam_line`:
   $\lambda$, minus the slope.
3. `T_final`: $F / \lambda$, the warming at which the imbalance is zero.
4. **The scatter**: the 20 years as black dots, warming across and imbalance up:
   `plt.scatter(x, y, color="black", s=14, label="one year")`, with `first_years["warming_c"]` as
   `x` and `first_years["imbalance_w_m2"]` as `y`. The line, drawn on to where it meets zero, and
   the labels are given.

**Use these names**, because the self-check looks for them: `F_line`, `lam_line` and `T_final`.

In [ ]:
# ← your answer here: replace every ... with code
line = ...      # a LinearRegression fitted on first_years: the warming column in, the imbalance as the answer
F_line = line.intercept_
slope = line.coef_[0]
lam_line = ...  # λ: minus the slope
T_final = ...   # the warming where N = 0: F / λ
print("F =", round(F_line, 2), "W per m²;  λ =", round(lam_line, 2), "W per m² per °C;  the imbalance reaches 0 at",
      round(T_final, 2), "°C")

plt.figure(figsize=(7, 4))
...  # the whole scatter line: first_years' warming across, its imbalance up, black dots of size 14, label "one year"
plt.plot([0, T_final], [F_line, 0], color="tab:red", label="the straight line, N = F − λT")
plt.axhline(0, color="0.5", lw=0.8)
plt.xlabel("warming T (°C)")
plt.ylabel("energy imbalance N (W per m²)")
plt.title(f"Imbalance against warming, first {len(first_years)} years")
plt.legend()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(F_line - 8.5008) < 0.01, "a straight line on first_years: the warming in, the imbalance as the answer"
assert abs(lam_line - 1.1442) < 0.005, "λ is minus the line's slope"
assert abs(T_final - 7.4296) < 0.01, "the warming where the imbalance is zero is F divided by λ"
print(f"✓ Your turn 1 — F {F_line:.2f} W/m², λ {lam_line:.2f}, the imbalance reaches 0 at {T_final:.2f} °C")

**If the line held, the warming would stop near 7.43 °C.** The line puts the forcing of
four times the CO2 at $F$ = 8.5 W m⁻², and each degree of warming sends back $\lambda$ =
1.14 W m⁻² more. The dots sit close to the line in these 20 years; whether it holds
after them, section 4 checks. And the line has no time in it: it says *how warm*, not *when*.

**Think.** In year 20 the warming is 4.87 °C, well short of 7.43 °C. What holds the
Earth back from warming faster?

**The kept energy has to warm something.** **Heat capacity.** C: how much energy, kept for a year, it takes to warm a part of the Earth by one degree; a big C warms slowly. On the same
stove a small pot boils in minutes and a big one takes much longer. The surface and the upper
ocean warm together, like one pot, or **one box**:

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week12c/one_box.png" width="480">

**How fast the box warms is the slope of its warming curve**, written $dT/dt$: the rise in $T$
(°C) over a short run of time (years), as a line's slope is its rise over its run. The box:

$$C\,\frac{dT}{dt} = F - \lambda\,T$$

$C$ is in W m⁻² years per °C. At first $T$ is 0, all of $F$ warms the box, and it warms fast; as $T$ grows,
$F - \lambda T$ shrinks and the warming slows.

**Stepping the equation.** Divide both sides by $C$: the slope is

$$\frac{dT}{dt} = \frac{F - \lambda\,T}{C}.$$

Take a run of time $\Delta t$ (say "delta t", a step of time). Over it the slope is about rise over
run, the change in $T$ over the step:

$$\frac{T_{\text{next}} - T}{\Delta t} \approx \frac{F - \lambda\,T}{C}
\qquad\text{so}\qquad
T_{\text{next}} = T + \frac{F - \lambda\,T}{C}\,\Delta t.$$

Start from $T = 0$ and apply it again and again: each step adds the slope times the step. With
$\Delta t$ = 1 year, the step is a year:

$$T_{\text{next year}} = T + \frac{F - \lambda\,T}{C}.$$

A year is a short run for a curve this slow, so yearly steps are close enough for these pictures.

### ✏️ Your turn 2

**Can one box warm like the climate model?**

**Why:** this loop is a **toy climate model**, the simplest there is. If it follows the 20 years,
we have our answer.

1. Complete the function `one_box(F, lam, C)`. The list `temps` starts at `[0.0]`: no warming at
   year 0. Each of the 150 years, `T` is the last warming in the list (given); **you write
   `T_next`**, the next year's warming, $T + (F - \lambda T)/C$; it is appended (given).
2. Given: `F_line` and `lam_line` from Your turn 1, and a figure of a small box, $C$ = 8,
   and a big box, $C$ = 30.

In the function `F` is $F$, `lam` is $\lambda$, `C` is $C$ and `T` is $T$, this year's warming.

**Use this name**, because the self-check looks for it: `one_box`.

In [ ]:
# ← your answer here: replace every ... with code
# first_years: CESM2's training years (section 1); F_line and lam_line: your line's F and λ (Your turn 1)


def one_box(F, lam, C):
    """Step the one-box equation through 150 years; the warming in years 1 to 150, °C."""
    temps = [0.0]
    for year in range(150):  # 150 years
        T = temps[-1]        # the last warming in the list
        T_next = ...         # T + (F − λT) / C
        temps.append(T_next)
    return temps[1:]


small_box = one_box(F_line, lam_line, 8)
big_box = one_box(F_line, lam_line, 30)
plt.figure(figsize=(7, 4))
plt.scatter(first_years["year"], first_years["warming_c"], color="black", s=14, label="CESM2")
plt.plot(first_years["year"], small_box[:n_train], color="tab:green", label="one box, C = 8")
plt.plot(first_years["year"], big_box[:n_train], color="tab:brown", label="one box, C = 30")
plt.xlabel("years after CO2 was quadrupled")
plt.ylabel("global warming (°C)")
plt.title(f"One box against the first {n_train} years")
plt.legend()
plt.show()
print("year 1:  small box", round(small_box[0], 2), "°C,  big box", round(big_box[0], 2), "°C")
print("year", n_train, ": small box", round(small_box[n_train - 1], 2), "°C,  big box", round(big_box[n_train - 1], 2), "°C")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(one_box(8.5, 1.14, 8)[0] - 8.5 / 8) < 1e-6, "the first year: nothing has warmed yet, so the step is F / C"
assert abs(one_box(8.5, 1.14, 8)[29] - 7.3821) < 0.01, "each year: the last warming in the list plus (F − λT) / C"
assert abs(one_box(8.5, 1.14, 8)[149] - 7.4561) < 0.01, "150 years, returned without the starting 0"
assert abs(one_box(5, 2, 30)[29] - 2.1845) < 0.01, "one_box must use its own F, lam and C, not numbers typed inside it"
print(f"✓ Your turn 2 — a box of C = 8 warms {small_box[0]:.2f} °C in year 1 and {small_box[n_train - 1]:.2f} °C by year {n_train}")

**No single box can do both.** The small box warms 1.06 °C in the first year, near the
climate model's 0.87, but it is already at 7.09 °C by year 20, against
4.87. The big box is closer by year 20 (4.02 °C) but far too slow at first
(0.28 °C in year 1). The climate warms like a small box at first and like a big one later.

**The ocean is two boxes.** **Deep ocean.** The cold water below a few hundred metres. It warms slowly, by taking heat from the upper ocean, and while it does the surface warms more slowly too. Picture the small pot standing
in a big tub of cold water: it warms fast at first, then keeps losing heat to the tub, so it warms
only slowly until the tub warms too (Held and others, 2010).

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week12c/two_box.png" width="480">

The upper box loses heat to space, $\lambda T$, and passes heat down, $\gamma\,(T - D)$, more
when the gap between the boxes is larger. The deep box keeps what it gets:

$$C\,\frac{dT}{dt} = F - \lambda\,T - \gamma\,(T - D) \qquad
C_{\text{deep}}\,\frac{dD}{dt} = \gamma\,(T - D)$$

- $D$: the deep ocean's warming, in °C; the table has no column for it;
- $\gamma$ (gamma): how many W m⁻² pass down for each degree the upper box is warmer;
- $C_{\text{deep}}$: the deep ocean's heat capacity, much bigger than $C$.

### ✏️ Your turn 3

**Do two boxes warm fast, then slow?**

**Why:** the same loop as Your turn 2, now with two boxes. Each box's next warming is its warming
now plus its slope times one year ($\Delta t$ = 1):

$$T_{\text{next year}} = T + \frac{F - \lambda\,T - \gamma\,(T - D)}{C}, \qquad
D_{\text{next year}} = D + \frac{\gamma\,(T - D)}{C_{\text{deep}}}.$$

1. Complete the function `two_box(F, lam, gamma, C, C_deep)`. Both boxes start at 0. Each of the
   150 years, **you write `T_next`**, the surface's next warming, and **`D_next`**, the deep
   ocean's, from the two equations above. Moving to the next year and saving `T` are given.
2. Given: `F_line` and `lam_line` from Your turn 1, and a guess for the rest: $\gamma$ =
   1, $C$ = 8, $C_{\text{deep}}$ = 100.

In the function `gamma` is $\gamma$, `C_deep` is $C_{\text{deep}}$, `T` is this year's surface
warming and `D` the deep ocean's.

**Use this name**, because the self-check looks for it: `two_box`.

In [ ]:
# ← your answer here: replace every ... with code
# F_line, lam_line: from Your turn 1
def two_box(F, lam, gamma, C, C_deep):
    """Step both boxes through 150 years, one year at a time; the surface warming T each year, °C."""
    T = 0.0                  # surface warming at year 0
    D = 0.0                  # deep-ocean warming at year 0
    surface = []
    for year in range(150):  # 150 years
        T_next = ...         # T + (F − λT − γ(T − D)) / C
        D_next = ...         # D + γ(T − D) / C_deep
        T = T_next           # move both boxes to the next year
        D = D_next
        surface.append(T)
    return surface


guess = two_box(F_line, lam_line, 1, 8, 100)  # γ, C and C_deep guessed
plt.figure(figsize=(7, 4))
plt.scatter(first_years["year"], first_years["warming_c"], color="black", s=14, label="CESM2")
plt.plot(first_years["year"], guess[:n_train], color="tab:purple", label="two boxes, a guess")
plt.xlabel("years after CO2 was quadrupled")
plt.ylabel("global warming (°C)")
plt.title(f"Two boxes against the first {n_train} years")
plt.legend()
plt.show()
print("two boxes, year 1:", round(guess[0], 2), "°C;  year 10:", round(guess[9], 2), "°C;  year", n_train, ":", round(guess[n_train - 1], 2), "°C")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(two_box(5, 2, 0, 8, 100)[29] - one_box(5, 2, 8)[29]) < 1e-6, "with gamma = 0 no heat goes down, so T_next is Your turn 2's: T + (F − λT) / C"
assert abs(two_box(5, 2, 1, 8, 100)[1] - 1.0156) < 0.001, "T_next takes away the heat passed down, γ(T − D), as well as λT"
assert abs(two_box(5, 2, 1, 8, 20)[19] - 1.9808) < 0.01, "D_next: the deep ocean's warming plus the heat it gets, γ(T − D), over C_deep"
print(f"✓ Your turn 3 — two boxes reach {guess[n_train - 1]:.2f} °C by year {n_train}")

**Two boxes have the right shape: fast, then slow.** This guess reaches 4.18 °C by year
20, against the climate model's 4.87: close, not right. Five numbers are five dials, and
turning them by hand until the curve fits would take all day.

**This is the job for the rest of the class.** A neural network will find the five numbers from the
20 years while drawing the curve, then follow it for 130 years with no data. A dedicated fitting
program could find them too; we learn the network's way because it also works for equations that
are hard to step, or where the data are scattered.

## 2. Can a network predict the next 130 years from the first 20?

**A neural network draws curves, not only lines.** It is built from small pieces called neurons,
loosely copied from nerve cells.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week12c/neuron_comparison.png" width="420">

*Top, a nerve cell: signals arrive at its dendrites, the cell body (soma) adds them up, and the
long axon passes a signal on to other cells. Bottom, the artificial neuron: each input $x$ times a
weight $w$, the products added up with a bias $b$, then an activation $\varphi$, an S-curve,
decides what to pass on. Figure via the [Kardashev Scale wiki](https://kardashev.fandom.com/wiki/Neural_network).*

**Neuron.** A weighted sum of its inputs plus a constant, then an activation: the piece a network is built out of. With inputs $x_1, x_2, \ldots$ its weighted sum is
$z = w_1 x_1 + w_2 x_2 + \ldots + b$, where $b$, the **bias**, is a constant. On its own that is a
straight line.

**Activation.** What a neuron does to its weighted sum before passing it on; tanh turns it into a smooth step between −1 and 1. Without it, a stack of straight lines is still one straight line. Ours is $\tanh(z)$: a smooth S-curve from −1 to 1, like
week 9's S-curve stretched to run from −1 to 1.

**Neural network.** Layers of neurons, each layer feeding the next; adding up many smooth steps lets the whole draw curves. Neurons side by side make a **layer**; each layer's
outputs are the next layer's inputs. Ours reads the year and gives the warming, through two layers
of 32 neurons.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week12c/network_neuron.png" width="760">

*A network built up a layer at a time: one neuron, then a layer of them, then our network of two
layers. Each line is a weight; each neuron also has a bias. Training changes all of them. Drawn for
this course.*

**Try it.** Open the [TensorFlow Playground](https://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2&seed=0.63924&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false) and press ▶. Watch the test loss (top
right) fall and the boundary between the orange and blue points bend. Then change one thing at a
time: add a hidden layer, add neurons, or switch the activation to *Linear* (the boundary stays
straight). This notebook's networks are fixed; that page is where you try settings.

**A network chooses its own weights by making a loss small.**

**Loss.** The one number the network is trying to make small. Ours is the **mean squared miss**, the one that fits a straight line.

**Gradient descent.** Nudge every weight a little in the direction that makes the loss smaller, and repeat. Like walking down a hill in fog: you cannot see
the bottom, but you feel which way the ground slopes, and you step downhill. Each step moves a
weight $w$ against the slope of the loss, its **gradient**:

$$w_{\text{new}} = w - \eta\,\frac{d\,\text{loss}}{dw}$$

**Learning rate.** How big each nudge of gradient descent is: too small and it crawls, too large and it overshoots. Here $\eta$ (eta) = 0.1.

**See it.** With many weights the loss is a landscape, and gradient descent walks down it:

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week12c/loss_landscape.jpg" width="460">

*A loss landscape: the height is the loss, the two directions across are two weights, and the
arrow is gradient descent walking downhill to a low point.* Welch Labs animates it in
[*The most cited paper of the century is a brilliant hack*, from 13:20](https://www.youtube.com/watch?v=QgH9sr7G13Q&t=800s).

**A simple case with a known answer:** the loss $(w - 3)^2$ is smallest at $w = 3$. Its gradient is
$2\,(w - 3)$ (calculus: the slope of $x^2$ is $2x$): −6 at $w = 0$, so stepping against it moves
$w$ up, toward 3.

### ✏️ Your turn 4

**Walk down the loss $(w - 3)^2$ by hand.**

**Why:** every network in this notebook learns by this one step. Write it once yourself before
PyTorch takes it over.

1. Given: `learning_rate`, $\eta$ = 0.1, set once before the loop; `w` starts at 0; the list
   `path` keeps every `w`; the loop takes 30 steps; `gradient` is the slope $2\,(w - 3)$ at this `w`.
2. **`w`**: one step downhill, $w - \eta \times$ gradient.

**Use these names**, because the self-check looks for them: `w` and `path`.

In [ ]:
# ← your answer here: replace every ... with code
learning_rate = 0.1         # η: how big each step is
w = 0.0
path = [w]
for step in range(30):
    gradient = 2 * (w - 3)  # the slope of (w − 3)² at this w
    w = ...                 # one step downhill: w minus learning_rate times the gradient
    path.append(w)
print("the first three steps:", round(path[1], 2), round(path[2], 2), round(path[3], 2), ";  after 30 steps: w =", round(w, 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(path[1] - 0.6) < 1e-9, "the first step: from w = 0 the gradient is −6, so w moves to 0 − learning_rate × (−6); rerun the whole cell, so w starts again at 0"
assert abs(path[2] - 1.08) < 1e-9, "the second step: the gradient at 0.6 is 2 × (0.6 − 3)"
assert abs(w - 3) < 0.01, "each step moves w against the gradient, by the learning rate times the gradient"
print(f"✓ Your turn 4 — 30 steps took w from 0 to {w:.3f}")

In [ ]:
# path: every w from Your turn 4
w_grid = np.arange(0, 4.01, 0.1)
plt.figure(figsize=(5, 3))
plt.plot(w_grid, (w_grid - 3) ** 2, color="0.5", label="the loss (w − 3)²")
plt.scatter(path, (np.array(path) - 3) ** 2, color="tab:red", s=15, label="the steps, from w = 0")
plt.xlabel("w")
plt.ylabel("loss")
plt.title("Gradient descent: 30 steps downhill")
plt.legend()
plt.show()

**The steps walk to 3**: 0.6, 1.08, 1.46, ..., then
2.996, each smaller as the slope flattens. PyTorch does the same with three lines a step,
and works out the gradient itself:

- `optimiser.zero_grad()` clears the old gradient (PyTorch adds each new one to the old otherwise);
- `loss.backward()` works out the gradient of the loss for every number being learned;
- `optimiser.step()` takes the step.

The **optimiser** takes the steps. Ours, **Adam**, is gradient descent that adapts each number's
step as it goes: a network has numbers of very different sizes, and one fixed $\eta$ would crawl
for some and overshoot for others. `torch.tensor(0.0, requires_grad=True)` is a number PyTorch
will learn: `requires_grad=True` asks it to track how the loss depends on that number.

**Now the same loop on the climate, with PyTorch doing the slope.**
PyTorch works on **tensors**, its own arrays, like numpy's. `to_tensor(values)`, below, makes a
column of numbers into a tensor with one row per year;
`dtype=torch.float32` is the precision networks use.

**Epoch.** One pass through all of the training data. From here each step of training uses all the data at once, so each
step is one epoch, and the loops count `epoch`.

In [ ]:
def to_tensor(values):
    """A column of numbers as a PyTorch tensor: one row per value, one column."""
    return torch.tensor(np.array(values), dtype=torch.float32).reshape(-1, 1)


T_data = to_tensor(first_years["warming_c"])           # T, the warming in the training years, °C
N_data = to_tensor(first_years["imbalance_w_m2"])      # N, the imbalance in the training years, W per m²
print("tensors of", T_data.shape[0], "rows and", T_data.shape[1], "column")

### ✏️ Your turn 5

**Can gradient descent find the line of Your turn 1 by itself?**

**Why:** gradient descent can learn the numbers of an equation, as section 3's network will. Here
it learns two whose answer you know: $F$ = 8.5 and $\lambda$ = 1.14.

⏱ **On Colab the answer cell takes about 10 seconds.**

1. Given: `F_gd` and `lam_gd` (*gd*: by gradient descent), both starting at 1, and Adam.
2. In the loop, **you write `miss`**, the line's miss every training year, $F - \lambda T - N$.
   The loss, the mean of `miss` squared, and the three lines around it are given.

`F_gd` is $F$, `lam_gd` is $\lambda$, `T_data` is $T$ and `N_data` is $N$.

**Use these names**, because the self-check looks for them: `F_gd` and `lam_gd`.

In [ ]:
# ← your answer here: replace every ... with code
# T_data, N_data: the training years as tensors (the cell above)
F_gd = torch.tensor(1.0, requires_grad=True)    # F, learned; starts at 1
lam_gd = torch.tensor(1.0, requires_grad=True)  # λ, learned; starts at 1
optimiser = torch.optim.Adam([F_gd, lam_gd], lr=0.1)
for epoch in range(1000):
    optimiser.zero_grad()                       # clear the old gradients
    miss = ...                                  # F − λT − N, every training year
    loss = (miss ** 2).mean()                   # the mean squared miss
    loss.backward()                             # the gradients of the loss for F and λ
    optimiser.step()                            # one step downhill
print("gradient descent: F =", round(F_gd.item(), 2), " λ =", round(lam_gd.item(), 2), ";  mean squared miss", round(loss.item(), 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(F_gd.item() - 8.5008) < 0.05 and abs(lam_gd.item() - 1.1442) < 0.01, "miss is F − λT − N, written with F_gd, lam_gd, T_data and N_data; rerun the whole cell, so F and λ start again at 1"
print(f"✓ Your turn 5 — gradient descent found F = {F_gd.item():.2f} and λ = {lam_gd.item():.2f}")

**The same 8.5 and 1.14 as the straight line**, found by walking downhill. A
network is trained the same way, with many more numbers.

**Scaling in and out**, as `StandardScaler` did for the basalts and the ocean water. A network trains best on numbers near 1: tanh
bends only near 0, and far from 0 it is flat, so its slope there is nearly 0 and gradient descent
barely moves. So we change units outside the network, not inside it:

- **in:** the year goes in as centuries, $t/100$, so the input runs from 0 to 1.5, not 0 to 150;
- **out:** the last layer is a plain weighted sum, with **no activation**, so it can give any
  number; we read its output in tens of degrees, so an output of 0.5 means 5 °C.

In symbols, with "net" the network itself: $\hat T = 10 \times \text{net}(t / 100)$.

In [ ]:
epochs = 3000                                 # how many times every training loop below goes through its data
years = to_tensor(cesm2["year"])                  # all 150 years, the network's input
print(len(T_data), "years to learn from;", len(years), "years the network draws")

**Reading the network line.** Your turn 6 builds its network in one line,

`net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))`,

which is the network of the figure above, read from left to right:

| piece | what it is in the figure |
|---|---|
| `nn.Linear(1, 32)` | layer 1: 32 neurons, each a weighted sum of the 1 input, the year, plus its bias |
| `nn.Tanh()` | tanh on each of those 32 sums |
| `nn.Linear(32, 32)` | layer 2: 32 neurons, each a weighted sum of the 32 outputs of layer 1, plus its bias |
| `nn.Tanh()` | tanh on each of those 32 |
| `nn.Linear(32, 1)` | the output: one weighted sum of the 32, plus a bias, with no activation: the warming |

`nn.Sequential(...)` runs the pieces in that order. In the [TensorFlow Playground](https://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2&seed=0.63924&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false),
the same shape is two hidden layers (the + and − buttons above the layers), though the Playground
allows at most 8 neurons in a layer. The physics-informed network of section 3 is this same line
with one change: its last layer gives two numbers, `nn.Linear(32, 2)`.

### ✏️ Your turn 6

**Train a network on the 20 years.**

**Why:** before adding any physics, see what a network does with the data alone.

⏱ **On Colab the answer cell trains for about 5 seconds.**

1. Given: **`net`**, a layer of 32 neurons reading 1 number, `nn.Tanh()`, a layer of 32 reading
   32, `nn.Tanh()`, and a layer reading 32 and giving 1 number; and Adam with learning rate 0.01,
   which `net.parameters()` hands every weight and bias. `epochs`, set in the cell above, is
   3,000. These settings were checked to work; the Playground is where you try others.
2. Given, in each epoch: `T_hat`, the network's warming $\hat T$ in all 150 years, and
   `loss = data_loss`.
3. **You write the other four lines of each epoch**, the training loop shown after Your turn 4:
   clear the old gradients; **`data_loss`**, the mean squared miss of `T_hat` against `T_data` in
   the training years, `T_hat[:n_train]`; the gradient of the loss; one step downhill. Every
   network in this notebook trains with these lines.

The print gives the **typical miss**, the root mean squared miss: the square root of the mean
squared miss, in °C, as in week 7.

**Use these names**, because the self-check looks for them: `net` and `data_loss`.

In [ ]:
# ← your answer here: replace every ... with code
# years: all 150 years; T_data: the warming in the training years (the cells above)
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))  # one output: T̂
optimiser = torch.optim.Adam(net.parameters(), lr=0.01)
for epoch in range(epochs):
    ...                             # 1. clear the old gradients
    output = 10 * net(years / 100)  # the network, all 150 years
    T_hat = output[:, :1]           # the warming T̂

    # DATA LOSS: match CESM2's warming in the training years
    data_loss = ...  # 2. the mean squared miss of T̂ against T_data, training years only

    loss = data_loss
    ...  # 3. the gradient of the loss for every weight
    ...  # 4. one step downhill
print("after", epochs, "epochs the typical miss is", round(data_loss.item() ** 0.5, 2), "°C")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert not isinstance(data_loss, type(...)), "fill in data_loss first: the mean squared miss of T̂ against T_data"
assert hasattr(data_loss, "backward"), "data_loss must stay a PyTorch tensor: no .item() on that line"
mse_now = (((10 * net(years / 100))[:n_train] - T_data) ** 2).mean().item()
assert mse_now < 0.1, "the network has not learned the training years: each epoch needs all four lines, optimiser.zero_grad(), data_loss, loss.backward() and optimiser.step(); rerun the whole cell from the seed"
assert mse_now < 0.03, "the network learned the wrong years: data_loss compares T_hat[:n_train], the first n_train years, with T_data"
assert data_loss.item() < 3 * mse_now + 1e-3, "data_loss: the mean of (T̂ − T_data) squared over the training years; not its sum, square root or absolute value"
print(f"✓ Your turn 6 — the network misses the training years by {data_loss.item() ** 0.5:.2f} °C, typically")

**A typical miss of about 0.05 °C** on the years it learned from. Now the test: the next cell runs the network on all 150 years and shows the
130 that were hidden.

In [ ]:
with torch.no_grad():                              # only run it: no training here
    plain_curve = (10 * net(years / 100)).numpy().ravel()

plt.figure(figsize=(7, 4))
plt.scatter(later["year"], later["warming_c"], color="0.7", s=10, label="hidden from training")
plt.scatter(first_years["year"], first_years["warming_c"], color="black", s=14, label=f"first {n_train} years: training data")
plt.plot(cesm2["year"], plain_curve, color="tab:orange", lw=2, label="plain network")
plt.axvline(n_train, color="black", ls=":")
plt.xlabel("years after CO2 was quadrupled")
plt.ylabel("global warming (°C)")
plt.title(f"CESM2: a plain network trained on {len(first_years)} years")
plt.legend(fontsize=8)
plt.show()
print("year 150: the climate model", round(cesm2["warming_c"].iloc[-1], 2), "°C;  the plain network", round(plain_curve[-1], 2), "°C")

**Right on the 20 years, wrong after them.** The climate model goes on warming, to 7.55 °C
by year 150. The network ends at 8.24 °C, too warm. Nothing in its loss says what
should happen after year 20, so outside its years the curve is whatever its weights happen to
give: here it climbs too far; from another start, or on another model, it might bend another way.
Fitting the data is not the same as knowing the physics.

## 3. Can a network that obeys the energy balance do better?

**Physics-informed network.** A network whose loss adds how badly its curve breaks a known equation, so it must fit the data and obey the physics at once. Like a student who must match
the answers at the back of the book and also obey the rules of algebra: the rules guide them where
the book gives no answer. Our rules are the two-box equations, which hold in every year, the
130 without data too.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week12c/pinn_two_box.png" width="760">

*Its loss has two parts. The **data loss** compares the network with CESM2 in the 20 years
we have; the **physics loss** checks the two-box equations in all 150 years. Drawn for this
course.*

**The same loop, one more loss.** Your turn 6's loss was a data loss. Your turn 7 keeps it and
adds a physics loss. Four things are new; the next cells take them one at a time.

1. **Two outputs**, $\hat T$ and $\hat D$: the last layer is `nn.Linear(32, 2)`.
2. **The data loss checks the imbalance too.** The network's imbalance is
   $\hat N = F - \lambda \hat T$, and its miss against CESM2's $N$ is added to the warming's.
   Both are needed: from the warming alone, the network cannot tell $F$ from $\lambda$.
3. **The physics loss**, from the equations' residuals (below).
4. **Five numbers learned with the weights**, $F, \lambda, \gamma, C, C_{\text{deep}}$ (below).

**3. The physics loss.** **Equation residual.** How far a curve is from obeying an equation: the left side minus the right side, zero when the equation holds. For each box, the left
side of its equation minus the right side, in every year:

$$r_{\text{surface}} = C\,\frac{d\hat T}{dt} - \left(F - \lambda \hat T - \gamma(\hat T - \hat D)\right),
\qquad r_{\text{deep}} = C_{\text{deep}}\,\frac{d\hat D}{dt} - \gamma(\hat T - \hat D).$$

Each is 0 in a year where the curves obey the equation. One more is the start, `r_start`: both
warmings at year 0, when nothing had warmed yet, so it too should be 0. The physics loss is the mean
of each residual squared, added up, and training pushes it toward 0.

The slopes $d\hat T/dt$ and $d\hat D/dt$ come from PyTorch: `torch.autograd.grad` gives the
slope of anything the network computed with respect to an input it tracks. **This is why the
network uses tanh:** a tanh network draws a smooth curve, so its slope is smooth too; a ReLU
network (one of the Playground's choices) draws straight pieces joined at corners, so its slope
jumps.

**4. Five positive numbers.** A heat capacity cannot be negative, nor can the others; learned
directly, one turned negative. So the network learns their logarithms, `log_numbers`, and uses
`torch.exp(log_numbers)`, $e$ to the power of each: always positive, and $e^0 = 1$, so each starts
at 1. The next cell tracks the 150 years, so PyTorch can give slopes with respect to them.

In [ ]:
# the same 150 years, now tracked: without requires_grad_, PyTorch has no slope to give and stops with an error
years.requires_grad_(True)
year_zero = to_tensor([0])                       # the moment CO2 jumps: nothing has warmed yet
print(len(T_data), "years of data;", len(years), "years where the equations are checked")

### ✏️ Your turn 7

**Train a network that must fit the 20 years and obey the two-box equations.**

**Why:** the plain network went wrong because nothing told it what happens after year 20. The
equations hold in every year, so a physics loss should keep the curve rising.

⏱ **On Colab the answer cell trains for about 10 seconds.**

The loop is Your turn 6's. Given: the network, Your turn 6's line with a last layer of two outputs
(`nn.Linear(32, 2)`, for $\hat T$ and $\hat D$), the five numbers, the
**data loss**, and the two slopes. **You write the network's imbalance and the physics loss:**

1. **`N_hat`**: the imbalance the network implies, $\hat N = F - \lambda \hat T$; the data loss
   compares it with the climate model's;
2. **`r_surface`** and **`r_deep`**: the two residuals of the box above, with `T_hat` for
   $\hat T$, `D_hat` for $\hat D$, `dT_dt` and `dD_dt` for the slopes, and `F`, `lam`, `gamma`,
   `C`, `C_deep` for the five numbers;
3. **`physics_loss`**: the mean of `r_surface` squared, plus the mean of `r_deep` squared, plus
   the mean of `r_start` squared (`r_start`, the start, is given).

`loss` adds the data loss and the physics loss. The self-check reads what they train, `pinn` and
`log_numbers`.

In [ ]:
# ← your answer here: replace every ... with code
# T_data, N_data: the training years as tensors; years: all 150 years, tracked (cells above)
torch.manual_seed(0)
pinn = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 2))  # two outputs: T̂ and D̂
log_numbers = torch.tensor([0.0, 0.0, 0.0, 0.0, 0.0], requires_grad=True)       # the logs of F, λ, γ, C, C_deep; each number starts at e⁰ = 1
optimiser = torch.optim.Adam(list(pinn.parameters()) + [log_numbers], lr=0.01)  # learns the weights and the five numbers
for epoch in range(epochs):
    optimiser.zero_grad()
    F, lam, gamma, C, C_deep = torch.exp(log_numbers)                           # e to each: always positive
    output = 10 * pinn(years / 100)                                             # the network, all 150 years
    T_hat = output[:, :1]                                                       # the surface warming T̂ (first column)
    D_hat = output[:, 1:]                                                       # the deep-ocean warming D̂ (second column)
    N_hat = ...                                                                 # F − λT̂: the network's imbalance N̂, every year

    # DATA LOSS: match CESM2's warming and (NEW) its imbalance in the training years
    data_loss = ((T_hat[:n_train] - T_data) ** 2).mean() + ((N_hat[:n_train] - N_data) ** 2).mean()

    # PHYSICS LOSS (NEW): obey the two-box equations in all 150 years
    dT_dt = torch.autograd.grad(T_hat.sum(), years, create_graph=True)[0]  # the slope of T̂, °C per year, every year
    dD_dt = torch.autograd.grad(D_hat.sum(), years, create_graph=True)[0]  # the slope of D̂, °C per year, every year
    r_surface = ...                                                        # C dT̂/dt − (F − λT̂ − γ(T̂ − D̂))
    r_deep = ...                                                           # C_deep dD̂/dt − γ(T̂ − D̂)
    r_start = 10 * pinn(year_zero / 100)                                   # the start: both warmings at year 0, which should be 0
    physics_loss = ...                                                     # the mean of each residual squared, all three added

    loss = data_loss + physics_loss
    loss.backward()
    optimiser.step()
F, lam, gamma, C, C_deep = torch.exp(log_numbers).detach().numpy()  # the five numbers after training
print("learned: F", round(F, 2), " λ", round(lam, 2), " C", round(C, 1), " C_deep", round(C_deep, 1))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert N_hat.shape[0] == len(years), "N_hat is F − λT̂ with the network's T_hat, one value for each of the 150 years, not the data's T"
assert abs(torch.exp(log_numbers)[1].item() - 1.194) < 0.05 and 15 < torch.exp(log_numbers)[4].item() < 80, "N_hat is F − λT̂; each residual is its equation's left side minus its right side, as in the box above; the physics loss is the mean of each squared, added; rerun the whole cell from the seed"
assert abs(torch.exp(log_numbers)[3].item() - 7.59) < 1.5 and 21.8 < torch.exp(log_numbers)[4].item() < 40.6, "r_surface is C dT̂/dt − (F − λT̂ − γ(T̂ − D̂)), with C; the physics loss adds the mean of r_start squared too"
assert 6.5 < 10 * pinn(torch.tensor([[1.5]]))[0, 0].item() < 7.8, "the network should keep warming after the training years; check both residuals and the physics loss"
print(f"✓ Your turn 7 — the network learned λ = {torch.exp(log_numbers)[1].item():.2f} and a deep ocean of C_deep = {torch.exp(log_numbers)[4].item():.1f}")

**Now the curves.** The next cell draws the physics-informed network's surface warming for all 150
years, beside the plain network's and the hidden years, and its deep-ocean warming, which no data
showed.

In [ ]:
with torch.no_grad():                             # only run it: no training here
    output = 10 * pinn(years / 100)
pinn_curve = output[:, :1].numpy().ravel()        # T̂, the surface
deep_curve = output[:, 1:].numpy().ravel()        # D̂, the deep ocean

plt.figure(figsize=(7, 4))
plt.scatter(later["year"], later["warming_c"], color="0.7", s=10, label="hidden from training")
plt.scatter(first_years["year"], first_years["warming_c"], color="black", s=14, label=f"first {n_train} years: training data")
plt.plot(cesm2["year"], plain_curve, color="tab:orange", lw=2, label="plain network: data loss only")
plt.plot(cesm2["year"], pinn_curve, color="tab:blue", lw=2, label="physics-informed network: data + physics loss")
plt.plot(cesm2["year"], deep_curve, color="navy", ls="--", label="physics-informed network, deep ocean")
plt.axvline(n_train, color="black", ls=":")
plt.xlabel("years after CO2 was quadrupled")
plt.ylabel("warming (°C)")
plt.title(f"CESM2: two networks trained on {len(first_years)} years")
plt.legend(fontsize=8)
plt.show()
print("year 150: the climate model", round(cesm2["warming_c"].values[-1], 2), "°C;  plain network", round(plain_curve[-1], 2),
      "°C;  physics-informed network", round(pinn_curve[-1], 2), "°C;  its deep ocean", round(deep_curve[-1], 2), "°C")
print("years 121-150 on average: the climate model", round(cesm2["warming_c"].values[120:].mean(), 2),
      "°C;  physics-informed network", round(pinn_curve[120:].mean(), 2), "°C")

**The physics-informed network stays close over the years it never saw**: 7.02 °C at year
150, against the climate model's 7.55 and the plain network's 8.24. Same 20
years of data; the difference is the physics loss, which held the curve to the two-box equations
in years 21 to 150 as well. The equations narrow the curves the network can draw; they
do not pick one exactly. Year 150 is a single, noisy year: over years 121 to 150 the climate model
averages 7.26 °C and the physics-informed network 6.93. Close is not
the same: the network levels off sooner, and in the last decades the climate model keeps rising
faster, so the gap grows toward year 150.

**In the network's picture, the deep ocean explains the slow rise.** The network learned $F$ =
8.69 and $\lambda$ = 1.19, close to the line's 8.5 and 1.14,
and a deep box 4 times bigger than the upper one ($C_{\text{deep}}$ =
29.0, $C$ = 7.6). Its deep ocean (dashed) warms slowly, drawing heat down fast at
first and little by the end, when it is at 6.71 °C, close to the surface. A box this size holds about as much heat as
the top couple of hundred metres of ocean, not the deep ocean: the network's picture is simpler
than the real one. Section 4 checks whether this picture is right.

**How far is each network from the 130 hidden years?** A figure can mislead the eye. The next cell
gives one number per network, the typical miss over years 21 to 150, as in Your turn 6. `.values`
takes a column out of the table as a numpy array.

In [ ]:
# later: the hidden years; plain_curve, pinn_curve: each network's warming, years 1-150 (cells above)
actual = later["warming_c"].values         # the climate model in the hidden years, °C: the column as a numpy array
rms_plain = np.sqrt(((plain_curve[n_train:] - actual) ** 2).mean())
rms_pinn = np.sqrt(((pinn_curve[n_train:] - actual) ** 2).mean())
print("typical miss over the hidden years: plain network", round(rms_plain, 2), "°C;  physics-informed network", round(rms_pinn, 2), "°C")

**From a typical miss of 1.05 °C to 0.21 °C.** The physics-informed network
is about 5.0 times closer to the 130 years it never saw.

## 4. Can we trust what the network learned?

**Is it right for the right reason?** The network's equation also says what the imbalance is,
$\hat N = F - \lambda \hat T$. The climate model's imbalance after year 20 was hidden too, so it
is a second test, of the physics inside the network rather than of its curve.

### ✏️ Your turn 8

**Does the network get the hidden imbalance right too?**

**Why:** a curve can match the warming with the wrong physics inside. If the network's equation is
right, its $F - \lambda \hat T$ should follow the climate model's $N$ after year 20 as well.

1. Given: `F_pinn` and `lam_pinn`, the $F$ and $\lambda$ the network learned, `N_all`, the
   climate model's imbalance in all 150 years, and `rms_N`, the typical miss.
2. **`implied_N`**: the network's imbalance in every year, $F - \lambda \hat T$, with
   `pinn_curve` as $\hat T$.
3. **`line_later`**, **`slope_later`** and **`lam_later`**: Your turn 1 again, on `later`, the
   hidden years: a straight line with the warming in and the imbalance as the answer, its slope,
   and $\lambda$, minus the slope. **Why:** does $\lambda$ stay the same after year 20?

**Use these names**, because the self-check looks for them: `implied_N`, `line_later`, `slope_later`
and `lam_later`.

In [ ]:
# ← your answer here: replace every ... with code
# pinn_curve: the network's warming, years 1-150, and log_numbers (Your turn 7); later: the hidden years;
# lam_line: λ of your line on the training years (Your turn 1)
F_pinn = torch.exp(log_numbers)[0].item()                               # the F the network learned, W per m²
lam_pinn = torch.exp(log_numbers)[1].item()                             # the λ it learned, W per m² per °C
N_all = cesm2["imbalance_w_m2"].values                                  # the climate model's imbalance, years 1-150, W per m²
implied_N = ...                                                         # the network's imbalance every year: its F, minus its λ times its warming
rms_N = np.sqrt(((implied_N[n_train:] - N_all[n_train:]) ** 2).mean())  # the typical miss, in the hidden years only
line_later = ...                                                        # a LinearRegression fitted on later, the hidden years: the warming column in, the imbalance as the answer
slope_later = ...                                                       # the slope of line_later, as slope in Your turn 1
lam_later = ...                                                         # λ of the hidden years: minus slope_later
print("year 150: the climate model's imbalance", round(N_all[-1], 2), "W per m²;  the network's", round(implied_N[-1], 2), "W per m²")
print("typical miss in the imbalance, hidden years:", round(rms_N, 2), "W per m²")
print("λ from a line on the training years:", round(lam_line, 2), ";  on the hidden years:", round(lam_later, 2), ";  the network's λ:", round(lam_pinn, 2))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert len(implied_N) == 150 and abs(implied_N[0] - (F_pinn - lam_pinn * pinn_curve[0])) < 1e-4 and abs(implied_N[-1] - (F_pinn - lam_pinn * pinn_curve[-1])) < 1e-4, "implied_N: the network's F minus its λ times its warming, every year"
assert abs(slope_later - -0.4128) < 0.005, "line_later: a straight line on later, the warming in and the imbalance as the answer; slope_later: its slope"
assert abs(lam_later - 0.4128) < 0.005, "lam_later is minus slope_later"
print(f"✓ Your turn 8 — the network misses the hidden imbalance by {rms_N:.2f} W/m², and λ falls from {lam_line:.2f} to {lam_later:.2f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(later["year"], later["imbalance_w_m2"], color="0.7", s=10, label="hidden from training")
plt.scatter(first_years["year"], first_years["imbalance_w_m2"], color="black", s=14, label=f"first {n_train} years: training data")
plt.plot(cesm2["year"], implied_N, color="tab:blue", lw=2, label="the network's imbalance, F − λT̂")
plt.axhline(0, color="0.5", lw=0.8)
plt.axvline(n_train, color="black", ls=":")
plt.xlabel("years after CO2 was quadrupled")
plt.ylabel("energy imbalance N (W per m²)")
plt.title("CESM2: the hidden imbalance and the network's")
plt.legend(fontsize=8)
plt.show()

**The network gets the warming right but not the imbalance.** At year 150 CESM2 is still gaining
2.36 W m⁻²; the network says 0.32. Over the hidden years its imbalance misses by
1.41 W m⁻², typically, and it falls too fast after year 20.

**A constant λ is too simple for CESM2.** A line through years 21 to 150 has $\lambda$ =
0.41, against 1.14 in the first 20 years: as CESM2 warms, each extra degree
sends less extra heat to space. **Pattern effect.** λ changing as the Earth warms, because the pattern of warming over the globe changes and some regions send heat to space more easily than others; in most climate models λ gets smaller with time. Andrews, Gregory and
Webb (2015) saw it in this same experiment in most of the CMIP5 climate models, the generation
before CESM2; much of it comes from the tropics, where the eastern Pacific warms late. The
network's equation holds $\lambda$ fixed, near 1.19, so it sends too much heat to space;
to match the warming anyway, it sends too little into its deep ocean. The two mistakes largely
cancel in the warming, until the last decades.

The next figure draws the pattern effect: every year of CESM2 as a point of warming against
imbalance, with your line on the first 20 years (Your turn 1) and your line on the hidden years
(Your turn 8). Where each line reaches zero imbalance is where its warming would stop, if that
line held. The second is a guess too: it reaches far beyond the warming CESM2 shows, and λ may
keep falling.

In [ ]:
# F_line, lam_line, T_final: your line on the training years (Your turn 1); lam_later: on the hidden years (Your turn 8)
F_later = line_later.intercept_                   # F of the hidden years' line, W per m²
T_final_later = F_later / lam_later               # where that line reaches zero imbalance, °C
T_grid = np.array([0, 14])                        # warmings of 0 and 14 °C: a straight line needs only its two ends
plt.figure(figsize=(7, 4.5))
plt.scatter(later["warming_c"], later["imbalance_w_m2"], color="0.7", s=10, label=f"years {n_train + 1}-150")
plt.scatter(first_years["warming_c"], first_years["imbalance_w_m2"], color="black", s=14, label=f"years 1-{n_train}")
plt.plot(T_grid, F_line - lam_line * T_grid, color="tab:red", label=f"line on years 1-{n_train}: λ = {lam_line:.2f}")
plt.plot(T_grid, F_later - lam_later * T_grid, color="tab:purple", label=f"line on years {n_train + 1}-150: λ = {lam_later:.2f}")
plt.axhline(0, color="0.5", lw=0.8)
plt.xlabel("global warming T (°C)")
plt.ylabel("energy imbalance N (W per m²)")
plt.title("CESM2: each extra degree sends less heat to space as the Earth warms")
plt.legend(fontsize=8)
plt.show()
print("each line reaches zero imbalance at: the training years", round(T_final, 2), "°C;  the hidden years",
      round(T_final_later, 2), "°C;  CESM2 has warmed", round(cesm2["warming_c"].values[-1], 2), "°C by year 150")

**The right warming partly hides a wrong mechanism.** The deep ocean of section 3 is the network's
explanation, not CESM2's. A physics-informed network is only as right as its equation: a right
answer does not prove the physics inside it.

## 5. Where do physics-informed networks earn their place?

**Where a trusted equation is known but hard to step, or where the data are scattered.** Our
climate problem could be stepped a year at a time, and a fitting program could do the same job.
Two Earth-science problems where an equation in the loss earns its keep:

- **How Antarctic ice shelves flow.** Satellites see how fast an ice shelf moves and how thick it
  is, not how stiff the ice is. Wang, Lai and others (*Science*, 2025) trained networks to match the
  satellite data and obey the equations of ice flow, and learned the flow law and stiffness maps of
  whole shelves.
- **How long a seismic wave takes to travel.** Locating an earthquake needs the travel time between
  any two points. EikoNet (Smith, Azizzadenesheli and Ross, 2021) trained a network on the
  travel-time equation for a known three-dimensional velocity model, with no travel times to learn
  from: the equation was the loss.

In both, as in ours, the answer is only as good as the equation in the loss.

## Week 12 summary

### The science

**If CO2 jumped to four times, how warm would Earth get, and when?**  
If CO2 jumped to four times, the CESM2 climate model warms 3.96 °C in 10 years, 4.87 °C in 20 and 7.55 °C in 150, and is still gaining 2.36 W m⁻²: fast for a few years, then slowly while the ocean takes up heat. Each degree of warming sends λ more heat to space, but CESM2's λ falls as it warms (1.14 in years 1-20, 0.41 after), so the first 20 years cannot say where the warming stops. This is one model's answer to a jump that will not happen, not a forecast for the real Earth. A network is trained by one loop: predict, measure the loss, take the gradient, step downhill. A physics-informed network is the same loop with a second loss: the data loss compares it with the data where there are data, and the physics loss, the squared residuals of the equations, holds it to the physics in every year, data or not. It learns the equations' numbers with its weights. From 20 years, a plain network goes wherever its weights give (8.24 °C at year 150, too warm) and the physics-informed one stays close (7.02 °C, against 7.55), but with the wrong imbalance (0.32 W m⁻² at year 150, against 2.36), because its equation holds λ fixed. A physics-informed network is only as right as its equation: a right answer does not prove the mechanism.

- **Climate model.** A computer program that steps the physics of the air, the ocean, ice and land forward in time, on a grid around the globe; its output is a simulation, not a measurement.
- **Energy imbalance.** N: the sunlight the Earth absorbs minus the heat it sends back to space, in watts per square metre; while N is above zero the Earth keeps gaining heat and keeps warming.
- **Radiative forcing.** F: the energy imbalance a change such as more CO2 causes before the Earth has warmed at all, in watts per square metre.
- **Climate feedback parameter.** λ: how many extra watts per square metre a warmer Earth sends back to space for each degree it has warmed.
- **Pattern effect.** λ changing as the Earth warms, because the pattern of warming over the globe changes and some regions send heat to space more easily than others; in most climate models λ gets smaller with time.
- **Heat capacity.** C: how much energy, kept for a year, it takes to warm a part of the Earth by one degree; a big C warms slowly.
- **Deep ocean.** The cold water below a few hundred metres. It warms slowly, by taking heat from the upper ocean, and while it does the surface warms more slowly too.
- **Neuron.** A weighted sum of its inputs plus a constant, then an activation: the piece a network is built out of.
- **Activation.** What a neuron does to its weighted sum before passing it on; tanh turns it into a smooth step between −1 and 1. Without it, a stack of straight lines is still one straight line.
- **Neural network.** Layers of neurons, each layer feeding the next; adding up many smooth steps lets the whole draw curves.
- **Loss.** The one number the network is trying to make small.
- **Gradient descent.** Nudge every weight a little in the direction that makes the loss smaller, and repeat.
- **Learning rate.** How big each nudge of gradient descent is: too small and it crawls, too large and it overshoots.
- **Epoch.** One pass through all of the training data.
- **Physics-informed network.** A network whose loss adds how badly its curve breaks a known equation, so it must fit the data and obey the physics at once.
- **Equation residual.** How far a curve is from obeying an equation: the left side minus the right side, zero when the equation holds.

### The code

| Function | What it does |
|---|---|
| **Python** | |
| `for x in things:` | do the same thing once for each item |
| `range(n)` | the numbers 0 to n-1, to count with |
| `list.append(x)` | add one item to the end of a list |
| `def name(a, b):` | write the recipe once and give it a name; the names in brackets are what it needs |
| `return a, b` | hand back two values at once; catch them with a, b = f(...) |
| `round(x, 1)` | trim a long decimal — 9.1 rather than sixteen digits |
| **NumPy** | |
| `np.sqrt(values)` | the square root |
| `np.exp(x)` | e to the power of x |
| **Matplotlib** | |
| `plt.plot(x, y)` | join points with a line: here, a warming curve against the year |
| `plt.scatter(x, y)` | a dot per pair of values |
| **scikit-learn** | |
| `LinearRegression().fit(x, y)` | find the straight line with the smallest total of squared residuals |
| `model.coef_[0]` | the slope of the fitted line |
| `model.intercept_` | the line's value when the input is 0: here F, the imbalance before any warming |
| **PyTorch** | |
| `torch.tensor(array)` | hand an array to PyTorch so it can be learned from |
| `torch.tensor(number, requires_grad=True)` | one number PyTorch will learn, as it learns a weight |
| `torch.manual_seed(n)` | fix the random start, so a training run repeats |
| `nn.Sequential(layers)` | a model that runs the layers you hand it, in order |
| `nn.Linear(in, out)` | one layer of weighted sums — a row of neurons |
| `nn.Tanh()` | the activation — a smooth step on each neuron, so a stack of layers can draw curves |
| `torch.optim.Adam(model.parameters(), lr=)` | the thing that nudges the weights downhill |
| `loss.backward() / optimiser.step() / optimiser.zero_grad()` | work out which way each weight should move, move them, then clear the slate |
| `net(x)` | run the network on its input: one output per row |
| `torch.autograd.grad(output.sum(), inputs)[0]` | the slope of an output at every input, worked out exactly by PyTorch |
| `torch.exp(t)` | e to the power of each value: always positive |
| `loss.item()` | one number out of a one-number tensor |
| `torch.no_grad()` | run the network without keeping the bookkeeping training needs |
| `tensor.numpy()` | take the numbers back out of PyTorch |

## Homework

**A second climate model.** MPI-ESM1.2-LR ran the same experiment; it is in `mpi`, with the same
columns. The homework repeats the three steps of class on it, and you write the code each time:
the straight line (Your turn 1), a plain network with the data loss only (Your turn 6), and a
physics-informed network with the data loss and the physics loss (Your turn 7). After a break, run
the setup cell and the checkpoint below; nothing else is needed.

Each part ends with questions under *What the numbers mean*. They are for you to think through in
words; nothing is typed there, and the self-checks grade only the code.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
epochs = 3000                                   # as in class


def to_tensor(values):
    """A column of numbers as a PyTorch tensor: one row per value, one column."""
    return torch.tensor(np.array(values), dtype=torch.float32).reshape(-1, 1)


n_train = 20                                     # the years to learn from, as in class
first_mpi = mpi[mpi["year"] <= n_train]          # MPI-ESM1.2-LR's training years
later_mpi = mpi[mpi["year"] > n_train]           # and the hidden ones
T_data_mpi = to_tensor(first_mpi["warming_c"])     # their warming, °C
N_data_mpi = to_tensor(first_mpi["imbalance_w_m2"])   # their imbalance, W per m²
years_mpi = to_tensor(mpi["year"])                # all 150 years, the networks' input
year_zero = to_tensor([0])                         # the moment CO2 jumps
print("ready:", len(first_mpi), "years of MPI-ESM1.2-LR to learn from")

### ✏️ Your turn 9

**Where does MPI-ESM1.2-LR's imbalance reach zero?**

**Why:** how warm first, with the straight line, as in class.

Repeat Your turn 1 on `first_mpi`.

1. `line_mpi`: the straight line, warming in, imbalance out;
2. `F_mpi`: its intercept, $F$; `slope_mpi`: its slope; `lam_mpi`: minus the slope, $\lambda$;
3. `T_final_mpi`: $F/\lambda$; the print is given.

**Use these names**, because the self-check looks for them: `F_mpi`, `lam_mpi` and `T_final_mpi`.

In [ ]:
# ← your answer here: replace every ... with code
line_mpi = ...     # the straight line on first_mpi: the warming column in, the imbalance as the answer
F_mpi = ...        # F: the line's intercept, W per m²
slope_mpi = ...    # the line's slope
lam_mpi = ...      # λ: minus the slope, W per m² per °C
T_final_mpi = ...  # the warming where N = 0: F / λ
print("MPI-ESM1.2-LR: F =", round(F_mpi, 2), " λ =", round(lam_mpi, 2), "; the imbalance reaches 0 at", round(T_final_mpi, 2),
      "°C;  the model's year 150:", round(mpi["warming_c"].iloc[-1], 2), "°C")
print("CESM2 in class: the imbalance reaches 0 at 7.43 °C;  year 150: 7.55 °C")

T_ends = np.array([0, T_final_mpi])  # from no warming to where the line reaches zero imbalance
plt.figure(figsize=(6, 4))
plt.scatter(first_mpi["warming_c"], first_mpi["imbalance_w_m2"], color="black", s=14, label="one year")
plt.plot(T_ends, F_mpi - lam_mpi * T_ends, color="tab:red", label="the line, N = F − λT")
plt.axhline(0, color="0.5", lw=0.8)
plt.xlabel("global warming T (°C)")
plt.ylabel("energy imbalance N (W per m²)")
plt.title(f"MPI-ESM1.2-LR: imbalance against warming, first {n_train} years")
plt.legend()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(F_mpi - 9.4017) < 0.01 and abs(lam_mpi - 1.6881) < 0.005, "a straight line on MPI-ESM1.2-LR's training years: the warming in, the imbalance out; λ is minus its slope"
assert abs(T_final_mpi - 5.57) < 0.01, "the warming where the imbalance is zero: F divided by λ"
print(f"✓ Your turn 9 — MPI-ESM1.2-LR: F {F_mpi:.2f}, λ {lam_mpi:.2f}, the imbalance reaches 0 at {T_final_mpi:.2f} °C")

**What the numbers mean.** How do F, λ and the final warming compare with CESM2's in class, and how far has each model got by
year 150? Which of F and λ makes the bigger difference? Think about what a larger λ means for a
warming planet.

### ✏️ Your turn 10

**Train a plain network on MPI-ESM1.2-LR's first 20 years.**

**Why:** the data loss alone, as in Your turn 6, on a second climate model.

⏱ **On Colab the answer cell trains for about 5 seconds.**

Given: `net_mpi`, the same network as Your turn 6, Adam, `T_hat` and `loss = data_loss`. **You
write the four lines of each epoch**, as in Your turn 6: clear the old gradients; `data_loss`, the
mean squared miss of `T_hat` against `T_data_mpi` in the training years; the gradient; one step
downhill. The cell after it draws the network against the hidden years.

**Use these names**, because the self-check looks for them: `net_mpi`, `data_loss` and `plain_mpi`.

In [ ]:
# ← your answer here: replace every ... with code
torch.manual_seed(0)
net_mpi = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))  # one output: T̂
optimiser = torch.optim.Adam(net_mpi.parameters(), lr=0.01)
for epoch in range(epochs):
    ...                                     # 1. clear the old gradients
    output = 10 * net_mpi(years_mpi / 100)  # the network, all 150 years
    T_hat = output[:, :1]                   # the warming T̂

    # DATA LOSS: match MPI-ESM1.2-LR's warming in the training years
    data_loss = ...  # 2. the mean squared miss of T̂ against T_data_mpi, training years only

    loss = data_loss
    ...                                                          # 3. the gradient of the loss for every weight
    ...                                                          # 4. one step downhill
with torch.no_grad():                                            # only run it: no training here
    plain_mpi = (10 * net_mpi(years_mpi / 100)).numpy().ravel()  # its warming, years 1-150
print("the plain network misses the training years by", round(data_loss.item() ** 0.5, 2), "°C, typically;  year 150:", round(plain_mpi[-1], 2), "°C")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert not isinstance(data_loss, type(...)), "fill in data_loss first: the mean squared miss of T̂ against T_data_mpi"
assert hasattr(data_loss, "backward"), "data_loss must stay a PyTorch tensor: no .item() on that line"
mse_now = (((10 * net_mpi(years_mpi / 100))[:n_train] - T_data_mpi) ** 2).mean().item()
assert mse_now < 0.1, "the network has not learned the training years: each epoch needs all four lines, optimiser.zero_grad(), data_loss, loss.backward() and optimiser.step(); rerun the whole cell from the seed"
assert mse_now < 0.03, "the network learned the wrong years: data_loss compares T_hat[:n_train], the first n_train years, with T_data_mpi"
assert data_loss.item() < 3 * mse_now + 1e-3, "data_loss: the mean of (T̂ − T_data_mpi) squared over the training years; not its sum, square root or absolute value"
print(f"✓ Your turn 10 — the plain network ends at {plain_mpi[-1]:.2f} °C in year 150")

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(later_mpi["year"], later_mpi["warming_c"], color="0.7", s=10, label="hidden from training")
plt.scatter(first_mpi["year"], first_mpi["warming_c"], color="black", s=14, label=f"first {n_train} years: training data")
plt.plot(mpi["year"], plain_mpi, color="tab:orange", lw=2, label="plain network: data loss only")
plt.axvline(n_train, color="black", ls=":")
plt.xlabel("years after CO2 was quadrupled")
plt.ylabel("global warming (°C)")
plt.title(f"MPI-ESM1.2-LR: a plain network trained on {n_train} years")
plt.legend(fontsize=8)
plt.show()

**What the numbers mean.** How well does the plain network fit the 20 years, and where does it end at year 150 compared with
the climate model? In class it climbed too far on CESM2: what in its loss would make it right or
wrong after year 20?

### ✏️ Your turn 11

**Train a physics-informed network on MPI-ESM1.2-LR, and test it.**

**Why:** the data loss and the physics loss, as in Your turn 7. In class the physics-informed
network was far closer on the warming, but wrong on the imbalance. Is it the same on a second
model?

⏱ **On Colab the answer cell trains for about 10 seconds.**

The loop is Your turn 7's, on `years_mpi`, `T_data_mpi` and `N_data_mpi`. **You write
`N_hat`**, the network's imbalance $F - \lambda \hat T$, **and both losses:** `data_loss` (the warming's miss plus the imbalance's miss, in the 20 years), the two
residuals `r_surface` and `r_deep`, and `physics_loss` (`r_start` is given). The cell after it draws both networks and
prints how far each is from the 130 hidden years.

**Use these names**, because the self-check looks for them: `pinn_mpi` and `log_numbers_mpi`.

In [ ]:
# ← your answer here: replace every ... with code
years_mpi.requires_grad_(True)                                                 # tracked, for the slopes
torch.manual_seed(0)
pinn_mpi = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 2))  # two outputs: T̂ and D̂
log_numbers_mpi = torch.tensor([0.0, 0.0, 0.0, 0.0, 0.0], requires_grad=True)  # the logs of F, λ, γ, C, C_deep; each starts at e⁰ = 1
optimiser = torch.optim.Adam(list(pinn_mpi.parameters()) + [log_numbers_mpi], lr=0.01)
for epoch in range(epochs):
    optimiser.zero_grad()
    F, lam, gamma, C, C_deep = torch.exp(log_numbers_mpi)                      # e to each: always positive
    output = 10 * pinn_mpi(years_mpi / 100)                                    # the network, all 150 years
    T_hat = output[:, :1]                                                      # the surface warming T̂
    D_hat = output[:, 1:]                                                      # the deep-ocean warming D̂
    N_hat = ...                                                                # F − λT̂: the network's imbalance N̂, every year

    # DATA LOSS: match MPI-ESM1.2-LR's warming and its imbalance in the training years
    data_loss = ...  # the mean squared miss of T̂ against T_data_mpi, plus that of N̂ against N_data_mpi, training years only

    # PHYSICS LOSS: obey the two-box equations in all 150 years
    dT_dt = torch.autograd.grad(T_hat.sum(), years_mpi, create_graph=True)[0]  # the slope of T̂, °C per year
    dD_dt = torch.autograd.grad(D_hat.sum(), years_mpi, create_graph=True)[0]  # the slope of D̂, °C per year
    r_surface = ...                                                            # C dT̂/dt − (F − λT̂ − γ(T̂ − D̂))
    r_deep = ...                                                               # C_deep dD̂/dt − γ(T̂ − D̂)
    r_start = 10 * pinn_mpi(year_zero / 100)                                   # the start: both warmings at year 0, which should be 0
    physics_loss = ...                                                         # the mean of each residual squared, all three added

    loss = data_loss + physics_loss
    loss.backward()
    optimiser.step()
F, lam, gamma, C, C_deep = torch.exp(log_numbers_mpi).detach().numpy()  # the five numbers after training
print("learned: F", round(F, 2), " λ", round(lam, 2), " C_deep", round(C_deep, 1))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert N_hat.shape[0] == len(years_mpi), "N_hat is F − λT̂ with the network's T_hat, one value for each of the 150 years, not the data's T"
assert abs(torch.exp(log_numbers_mpi)[1].item() - 1.63) < 0.1, "N_hat is F − λT̂; the data loss adds the warming's and the imbalance's mean squared miss in the training years; each residual is its equation's left side minus its right side; rerun the whole cell from the seed"
assert abs(torch.exp(log_numbers_mpi)[3].item() - 6.66) < 1.5 and 20.4 < torch.exp(log_numbers_mpi)[4].item() < 43.8, "each residual is its equation's left side minus its right side, C dT̂/dt − (F − λT̂ − γ(T̂ − D̂)) and C_deep dD̂/dt − γ(T̂ − D̂); the physics loss is the mean of each squared, all three added; rerun the whole cell from the seed"
assert abs(10 * pinn_mpi(torch.tensor([[1.5]]))[0, 0].item() - 5.54) < 0.3, "train from seed 0 with the data loss plus the physics loss, as in Your turn 7"
print(f"✓ Your turn 11 — the network learned λ = {torch.exp(log_numbers_mpi)[1].item():.2f}")

In [ ]:
with torch.no_grad():                                   # only run it: no training here
    pinn_mpi_curve = (10 * pinn_mpi(years_mpi / 100))[:, 0].numpy()   # its surface warming, years 1-150
F_mpi_net = torch.exp(log_numbers_mpi)[0].item()       # the F it learned
lam_mpi_net = torch.exp(log_numbers_mpi)[1].item()     # the λ it learned
implied_N_mpi = F_mpi_net - lam_mpi_net * pinn_mpi_curve   # its imbalance, every year
actual_mpi = mpi["warming_c"].values                  # the climate model's warming, years 1-150
N_all_mpi = mpi["imbalance_w_m2"].values              # its imbalance, years 1-150

plt.figure(figsize=(7, 4))
plt.scatter(later_mpi["year"], later_mpi["warming_c"], color="0.7", s=10, label="hidden from training")
plt.scatter(first_mpi["year"], first_mpi["warming_c"], color="black", s=14, label=f"first {n_train} years: training data")
plt.plot(mpi["year"], plain_mpi, color="tab:orange", lw=2, label="plain network: data loss only")
plt.plot(mpi["year"], pinn_mpi_curve, color="tab:blue", lw=2, label="physics-informed network: data + physics loss")
plt.axvline(n_train, color="black", ls=":")
plt.xlabel("years after CO2 was quadrupled")
plt.ylabel("global warming (°C)")
plt.title(f"MPI-ESM1.2-LR: two networks trained on {n_train} years")
plt.legend(fontsize=8)
plt.show()
print("year 150: the climate model", round(actual_mpi[-1], 2), "°C;  plain network", round(plain_mpi[-1], 2),
      "°C;  physics-informed network", round(pinn_mpi_curve[-1], 2), "°C")
print("typical miss over the hidden years: plain", round(np.sqrt(((plain_mpi[n_train:] - actual_mpi[n_train:]) ** 2).mean()), 2),
      "°C;  physics-informed", round(np.sqrt(((pinn_mpi_curve[n_train:] - actual_mpi[n_train:]) ** 2).mean()), 2), "°C")
plt.figure(figsize=(7, 4))
plt.scatter(later_mpi["year"], later_mpi["imbalance_w_m2"], color="0.7", s=10, label="hidden from training")
plt.scatter(first_mpi["year"], first_mpi["imbalance_w_m2"], color="black", s=14, label=f"first {n_train} years: training data")
plt.plot(mpi["year"], implied_N_mpi, color="tab:blue", lw=2, label="the network's imbalance, F − λT̂")
plt.axhline(0, color="0.5", lw=0.8)
plt.axvline(n_train, color="black", ls=":")
plt.xlabel("years after CO2 was quadrupled")
plt.ylabel("energy imbalance N (W per m²)")
plt.title("MPI-ESM1.2-LR: the hidden imbalance and the network's")
plt.legend(fontsize=8)
plt.show()
print("year 150 imbalance: the climate model", round(N_all_mpi[-1], 2), "W per m²;  the network", round(implied_N_mpi[-1], 2),
      "W per m²;  typical miss, hidden years:", round(np.sqrt(((implied_N_mpi[n_train:] - N_all_mpi[n_train:]) ** 2).mean()), 2), "W per m²")

**What the numbers mean.** Which network is closer to MPI-ESM1.2-LR's hidden years, and is that the same as in class? Is the
network's imbalance at year 150 too high or too low, and what does that say about its equation?
What would you need before trusting either network's answer for a third climate model?